# EDA

**how many samples, images, how big are they, how many classes, and is the data set balanced?**

We read the `.bin` files directly with numpy, because the format
is just raw `uint8` bytes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "data/stl10_binary"

IMG_SIZE = 96
CHANNELS = 3

# labels are 1..10, one name per line
class_names = [l.strip() for l in open(f"{DATA}/class_names.txt") if l.strip()]
print(f"{len(class_names)} classes:", ", ".join(class_names))

### How many images (labeled) ?

In [ ]:
import os

rows = []
for split, x_file, y_file in [
        ("train",     "train_X.bin",     "train_y.bin"),
        ("test",      "test_X.bin",      "test_y.bin"),
        ("unlabeled", "unlabeled_X.bin", None)]:
    n_bytes = os.path.getsize(f"{DATA}/{x_file}")
    n_images = n_bytes // (CHANNELS * IMG_SIZE * IMG_SIZE)
    rows.append({
        "split": split,
        "images": n_images,
        "labelled": y_file is not None,
        "size on disk (MB)": round(n_bytes / 1e6, 1),
    })

splits = pd.DataFrame(rows)
splits

In [ ]:
labelled = splits.loc[splits["labelled"], "images"].sum()
unlabelled = splits.loc[~splits["labelled"], "images"].sum()
total = labelled + unlabelled

print(f"labelled   : {labelled:,}   ({labelled / total:.1%} of the data)")
print(f"unlabelled : {unlabelled:,}   ({unlabelled / total:.1%})")
print(f"total      : {total:,}")

### What does one image look like?

Read the labelled set. The binaries store images **channel-first** as `(3, 96, 96)`

Lets transpose

In [ ]:
X_train = np.fromfile(f"{DATA}/train_X.bin", dtype=np.uint8).reshape(-1, CHANNELS, IMG_SIZE, IMG_SIZE)
y_train = np.fromfile(f"{DATA}/train_y.bin", dtype=np.uint8)

print(f"X_train : {X_train.shape}  {X_train.dtype}")
print(f"y_train : {y_train.shape}  {y_train.dtype}  values {y_train.min()}..{y_train.max()}")
print(f"pixel range: {X_train.min()}..{X_train.max()}")
print(f"memory    : {X_train.nbytes / 1e6:.0f} MB")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for ax, label in zip(axes.ravel(), range(1, 11)):
    idx = np.where(y_train == label)[0][0]
    img = X_train[idx].transpose(2, 1, 0)      # (3, 96, 96) -> (96, 96, 3)
    ax.imshow(img)
    ax.set_title(class_names[label - 1], fontsize=20)
    ax.axis("off")

plt.suptitle("One example per class (96x96 colour images)")
plt.tight_layout()
plt.show()

### Is it balanced?

Count the labels per class, separately for train and test.

In [ ]:
y_test = np.fromfile(f"{DATA}/test_y.bin", dtype=np.uint8)

balance = pd.DataFrame({
    "train": np.bincount(y_train, minlength=11)[1:],
    "test": np.bincount(y_test, minlength=11)[1:],
}, index=class_names)

balance.loc["total"] = balance.sum()
balance

### Summary

* **10 classes**: airplane, bird, car, cat, deer, dog, horse, monkey, ship, truck.
* **Images are 96 × 96 colour**, `uint8` (0–255)
* **13,000 labelled images** (5,000 train + 8,000 test) and **100,000 unlabelled** thus only
  **11.5%** of the dataset carries a label.
* The labelled part is **perfectly balanced**
* `unlabeled_X.bin` is 2.7 GB.